In [ ]:
import os
import calendar
import numpy as np
import netCDF4 as nc
import cfgrib
import sqlite3

def seconds_in_month(year, month):
    """
    Calculate the number of seconds in a given month of a specific year.

    Parameters
    ----------
    year (int): The year for which the month is being calculated.
    month (int): The month for which the number of seconds is being calculated (1-12).

    Returns
    ----------
    int: The number of seconds in the specified month.

    Raises
    ----------
    ValueError: If the month is not between 1 and 12.
    """
    # Check if the month is between 1 and 12
    if month < 1 or month > 12:
        raise ValueError("ERROR: Month must be between 1 and 12.")
    
    # Get the number of days in the specified month and year
    num_days = calendar.monthrange(year, month)[1]
    
    # Convert the number of days to seconds (days * 24 hours * 60 minutes * 60 seconds)
    return num_days * 24 * 60 * 60

def calculate_grid_cell_areas(lon, lat):
    """
    Calculate the area of each grid cell given latitude and longitude arrays.

    Parameters
    ----------
    lon (array-like): 1D array of longitudes in degrees.
    lat (array-like): 1D array of latitudes in degrees.

    Returns
    ----------
    numpy.ndarray: A 2D array of grid cell areas in square meters.

    Raises
    ----------
    ValueError: If lat or lon are not 1D arrays.
    """
    
    # Check that lon and lat are 1D arrays
    if len(lon.shape) != 1 or len(lat.shape) != 1:
        raise ValueError("ERROR: Both 'lon' and 'lat' must be 1D arrays.")
    
    # Radius of Earth in meters
    R = 6371000.0
    
    # Convert latitude to radians
    lat_rad = np.radians(lat)
    
    # Calculate grid cell width in radians
    dlat = np.radians(lat[1] - lat[0])
    dlon = np.radians(lon[1] - lon[0])
    
    # Initialize the area array
    area = np.zeros((len(lat), len(lon)))
    
    # Calculate area of each grid cell in square meters
    for i in range(len(lat)):
        for j in range(len(lon)):
            area[i, j] = R**2 * dlat * dlon * np.cos(lat_rad[i])
    
    return area

def calculate_evaporation_rate(temperature_K, latent_heat_flux):
    """
    Convert latent heat flux to evaporation rate.

    Parameters
    ----------
    temperature_K : float or array
        Air temperature in Kelvin.

    latent_heat_flux_W_m2 : float or array
        Latent heat flux in W/m².

    Notes:
    No input validation is performed. Negative ``temperature_K`` or
    ``latent_heat`` values will produce numerically valid but physically
    meaningless results — callers are responsible for sanity-checking inputs.
    Sub-optimal: a future revision should raise ``ValueError`` for inputs
    outside physical ranges.
    
    Returns
    -------
    evaporation_rate_mm_s : float or array
        Evaporation rate in mm/s.
    """

    # Convert temperature to Celsius
    temperature_C = temperature_K - 273.15

    # Latent heat of vaporization (MJ/kg)
    lambda_MJ_kg = 2.501 - 0.002361 * temperature_C

    # Convert W/m² -> MJ/(s·m²)
    latent_heat_flux_MJ = latent_heat_flux * 1e-6

    # MJ/(s·m²) / MJ/kg = kg/(m²·s) = mm/s
    evaporation_rate_mm_s = latent_heat_flux_MJ / lambda_MJ_kg

    return evaporation_rate_mm_s

class CFSProcessing:
    """
    Process CFS GRIB files and calculate lake/land averaged variables.
    """

    # ------------------------------------------------------------------
    # Processor lookup
    # ------------------------------------------------------------------

    PROCESSORS = {
        "mean": "process_mean",
        "accumulated_depth": "process_accumulated_depth",
        "rate_to_accumulation": "process_rate_to_accumulation",
        "evaporation": "process_evaporation",
    }

    # ------------------------------------------------------------------
    # Initialization
    # ------------------------------------------------------------------

    def __init__(self, database, table):
            """
            Initialize database connection and ensure table exists.
            """
            self.database = database
            self.table = table
            self._initialize_database()

            self.lake_lookup = {
                        "eri": "erie",
                        "ont": "ontario",
                        "sup": "superior",
                        "mih": "michigan-huron",
                    }
    
    def _initialize_database(self):
        """
        Ensure database file exists.
        """
        conn = sqlite3.connect(self.database)
        conn.close()

    # ==================================================================
    # Main processing function
    # ==================================================================

    import os
import calendar
import numpy as np
import netCDF4 as nc
import cfgrib


class CFSProcessing:
    """
    Process CFS GRIB files and insert lake/land-averaged variables
    into the database.

    The variables to process are supplied by the user through the
    ``variables`` argument in ``process_files()``. This class does
    not contain any hardcoded CFS variable definitions.
    """

    def __init__(self, db):
        """
        Parameters
        ----------
        db : object
            Database object with an ``add()`` method.
        """

        self.db = db

        # Mapping from mask-file lake abbreviations to database names.
        self.lake_lookup = {
            "eri": "erie",
            "ont": "ontario",
            "sup": "superior",
            "mih": "michigan-huron",
        }

        # Available processing methods.
        #
        # The user specifies one of these names in the variable
        # configuration, e.g.:
        #
        # "processor": "mean"
        #
        self.processors = {
            "mean": self.process_mean,
            "accumulated_depth": self.process_accumulated_depth,
            "rate_to_accumulation": self.process_rate_to_accumulation,
        }

    # ==================================================================
    # Main processing function
    # ==================================================================

    def process_files(
        self,
        download_dir,
        mask_file,
        mask_variables,
        variables,
    ):
        """
        Process CFS GRIB files and insert lake/land-averaged variables
        into the database.

        Parameters
        ----------
        download_dir : str
            Directory containing downloaded CFS GRIB files.

        mask_file : str
            Path to the NetCDF mask file. The mask file must include:

                - latitude
                - longitude
                - lake/land mask variables

        mask_variables : list of str
            Mask variables to process.

            Expected format:

                "{lake_abbreviation}_{surface_type}"

            Examples:

                "sup_lake"
                "sup_land"
                "mih_lake"
                "eri_land"

        variables : dict
            Dictionary defining the CFS variables to process.

            Example:

                variables = {
                    "watr": {
                        "output_name": "runoff",
                        "long_name": "Runoff",
                        "units": "kg/m2",
                        "type_of_level": "surface",
                        "level": None,
                        "processor": "accumulated_depth",
                    },

                    "avg_sp": {
                        "output_name": "surface_pressure",
                        "long_name": "Surface Pressure",
                        "units": "Pa",
                        "type_of_level": "surface",
                        "level": None,
                        "processor": "mean",
                    },
                }

            Each variable configuration must contain:

                - output_name
                - long_name
                - units
                - type_of_level
                - level
                - processor

        Notes
        -----
        Variables are automatically grouped by their GRIB
        ``typeOfLevel`` and ``level`` so that each GRIB level is opened
        only once per file.
        """

        # --------------------------------------------------------------
        # Validate inputs
        # --------------------------------------------------------------

        self._validate_inputs(
            download_dir=download_dir,
            mask_file=mask_file,
            mask_variables=mask_variables,
            variables=variables,
        )

        # --------------------------------------------------------------
        # Load mask grid
        # --------------------------------------------------------------

        mask_ds = nc.Dataset(mask_file)

        try:

            mask_lat = mask_ds.variables["latitude"][:]
            mask_lon = mask_ds.variables["longitude"][:]

            # Calculate grid-cell areas.
            area = calculate_grid_cell_areas(
                mask_lon,
                mask_lat,
            )

            # ----------------------------------------------------------
            # Remove stale cfgrib index files
            # ----------------------------------------------------------

            self._remove_index_files(
                download_dir
            )

            # ----------------------------------------------------------
            # Group variables by GRIB level
            # ----------------------------------------------------------

            variable_groups = self._group_variables_by_level(
                variables
            )

            # ----------------------------------------------------------
            # Process each GRIB file
            # ----------------------------------------------------------

            for filename in sorted(
                os.listdir(download_dir)
            ):

                file = os.path.join(
                    download_dir,
                    filename,
                )

                # ------------------------------------------------------
                # Check file type
                # ------------------------------------------------------

                if not (
                    filename.startswith("flxf")
                    and filename.endswith(".grib.grb2")
                ):

                    print(
                        f"Skipping unrecognized file: "
                        f"{filename}"
                    )

                    continue

                # ------------------------------------------------------
                # Parse filename
                # ------------------------------------------------------

                try:

                    cfs_run, forecast_year, forecast_month = (
                        self._parse_filename(filename)
                    )

                except (IndexError, ValueError) as e:

                    print(
                        f"ERROR parsing filename "
                        f"{filename}: {e}. "
                        f"Skipping file."
                    )

                    continue

                # ------------------------------------------------------
                # Number of days in forecast month
                # ------------------------------------------------------

                _, num_days = calendar.monthrange(
                    forecast_year,
                    forecast_month,
                )

                print(
                    f"\nProcessing {filename}"
                )

                # ------------------------------------------------------
                # Open each GRIB level only once
                # ------------------------------------------------------

                for level_key, level_variables in (
                    variable_groups.items()
                ):

                    type_of_level, level = level_key

                    print(
                        f"  Opening "
                        f"typeOfLevel={type_of_level}, "
                        f"level={level}"
                    )

                    try:

                        ds = self._open_grib_level(
                            file=file,
                            type_of_level=type_of_level,
                            level=level,
                        )

                    except Exception as e:

                        print(
                            f"ERROR opening "
                            f"{filename} "
                            f"for {type_of_level}, "
                            f"level={level}: {e}"
                        )

                        continue

                    try:

                        # --------------------------------------------------
                        # Process all variables in this GRIB level
                        # --------------------------------------------------

                        for variable_name in level_variables:

                            config = variables[
                                variable_name
                            ]

                            self._process_variable(
                                ds=ds,
                                variable_name=variable_name,
                                config=config,
                                mask_ds=mask_ds,
                                mask_variables=mask_variables,
                                mask_lat=mask_lat,
                                mask_lon=mask_lon,
                                area=area,
                                num_days=num_days,
                                cfs_run=cfs_run,
                                forecast_year=forecast_year,
                                forecast_month=forecast_month,
                            )

                    finally:

                        # Always close the GRIB dataset.
                        try:
                            ds.close()
                        except Exception:
                            pass

        finally:

            # Always close the mask dataset.
            mask_ds.close()

    # ==================================================================
    # Input validation
    # ==================================================================

    def _validate_inputs(
        self,
        download_dir,
        mask_file,
        mask_variables,
        variables,
    ):
        """
        Validate inputs to ``process_files()``.
        """

        if not os.path.isdir(download_dir):
            raise ValueError(
                "ERROR: The specified download directory "
                "does not exist."
            )

        if not os.path.exists(mask_file):
            raise ValueError(
                "ERROR: mask_file not found."
            )

        if not isinstance(mask_variables, list):
            raise ValueError(
                "ERROR: mask_variables must be a list."
            )

        if not isinstance(variables, dict):
            raise ValueError(
                "ERROR: variables must be a dictionary."
            )

        required_keys = {
            "output_name",
            "long_name",
            "units",
            "type_of_level",
            "level",
            "processor",
        }

        for variable_name, config in variables.items():

            if not isinstance(config, dict):
                raise ValueError(
                    f"Configuration for '{variable_name}' "
                    f"must be a dictionary."
                )

            missing = (
                required_keys
                - config.keys()
            )

            if missing:
                raise ValueError(
                    f"Variable '{variable_name}' "
                    f"is missing configuration keys: "
                    f"{sorted(missing)}"
                )

            processor = config["processor"]

            if processor not in self.processors:
                raise ValueError(
                    f"Unknown processor '{processor}' "
                    f"for variable '{variable_name}'. "
                    f"Available processors: "
                    f"{list(self.processors.keys())}"
                )

    # ==================================================================
    # Remove cfgrib index files
    # ==================================================================

    @staticmethod
    def _remove_index_files(download_dir):
        """
        Remove existing cfgrib .idx files from the download directory.
        """

        for filename in os.listdir(
            download_dir
        ):

            if filename.endswith(".idx"):

                idx_file = os.path.join(
                    download_dir,
                    filename,
                )

                os.remove(idx_file)

    # ==================================================================
    # Parse CFS filename
    # ==================================================================

    @staticmethod
    def _parse_filename(filename):
        """
        Parse CFS forecast information from a filename.

        Expected filename structure:

            flxf.01.YYYYMMDDHH.YYYYMM....

        Returns
        -------
        tuple
            cfs_run
            forecast_year
            forecast_month
        """

        parts = filename.split(".")

        cfs_run = parts[2]

        forecast_year = int(
            parts[3][:4]
        )

        forecast_month = int(
            parts[3][4:6]
        )

        return (
            cfs_run,
            forecast_year,
            forecast_month,
        )

    # ==================================================================
    # Group variables by GRIB level
    # ==================================================================

    @staticmethod
    def _group_variables_by_level(
        variables
    ):
        """
        Group variables that share the same GRIB
        ``typeOfLevel`` and ``level``.

        Example
        -------
        Input:

            {
                "watr": {
                    "type_of_level": "surface",
                    "level": None,
                    ...
                },

                "avg_sp": {
                    "type_of_level": "surface",
                    "level": None,
                    ...
                },

                "avg_2t": {
                    "type_of_level": "heightAboveGround",
                    "level": 2,
                    ...
                },
            }

        Output:

            {
                ("surface", None): [
                    "watr",
                    "avg_sp",
                ],

                ("heightAboveGround", 2): [
                    "avg_2t",
                ],
            }
        """

        groups = {}

        for variable_name, config in (
            variables.items()
        ):

            key = (
                config["type_of_level"],
                config["level"],
            )

            if key not in groups:
                groups[key] = []

            groups[key].append(
                variable_name
            )

        return groups

    # ==================================================================
    # Open a GRIB level
    # ==================================================================

    @staticmethod
    def _open_grib_level(
        file,
        type_of_level,
        level,
    ):
        """
        Open a GRIB file for a specific
        typeOfLevel/level combination.

        If level is None, only typeOfLevel is used
        in the cfgrib filter.
        """

        filter_by_keys = {
            "typeOfLevel": type_of_level,
        }

        if level is not None:
            filter_by_keys["level"] = level

        return cfgrib.open_dataset(
            file,
            engine="cfgrib",
            filter_by_keys=filter_by_keys,
            decode_timedelta=False,
        )

    # ==================================================================
    # Process one variable
    # ==================================================================

    def _process_variable(
        self,
        ds,
        variable_name,
        config,
        mask_ds,
        mask_variables,
        mask_lat,
        mask_lon,
        area,
        num_days,
        cfs_run,
        forecast_year,
        forecast_month,
    ):
        """
        Extract and process one variable from an
        already-open GRIB dataset.
        """

        print(
            f"    Processing "
            f"{variable_name} "
            f"({config['long_name']})"
        )

        # --------------------------------------------------------------
        # Get variable from GRIB dataset
        # --------------------------------------------------------------

        if variable_name not in ds:

            print(
                f"    '{variable_name}' not found "
                f"in GRIB file. Skipping."
            )

            return

        data = ds[
            variable_name
        ]

        # --------------------------------------------------------------
        # Cut data to mask extent
        # --------------------------------------------------------------

        data_cut = data.sel(
            latitude=slice(
                mask_lat.max(),
                mask_lat.min(),
            ),
            longitude=slice(
                mask_lon.min(),
                mask_lon.max(),
            ),
        )

        # --------------------------------------------------------------
        # Remap data to mask grid
        # --------------------------------------------------------------

        data_remap = data_cut.interp(
            latitude=mask_lat,
            longitude=mask_lon,
            method="linear",
        )

        # --------------------------------------------------------------
        # Get processing function
        # --------------------------------------------------------------

        processor_name = config[
            "processor"
        ]

        processor = self.processors[
            processor_name
        ]

        # --------------------------------------------------------------
        # Process each mask
        # --------------------------------------------------------------

        for mask_var in mask_variables:

            # ----------------------------------------------------------
            # Load mask
            # ----------------------------------------------------------

            mask = mask_ds.variables[
                mask_var
            ][:]

            # ----------------------------------------------------------
            # Process variable
            # ----------------------------------------------------------

            value = processor(
                data=data_remap,
                mask=mask,
                area=area,
                num_days=num_days,
            )

            # ----------------------------------------------------------
            # Determine lake and surface type
            # ----------------------------------------------------------

            try:

                lake_abv, surface_type = (
                    mask_var.split("_", 1)
                )

            except ValueError:

                raise ValueError(
                    f"Invalid mask variable "
                    f"'{mask_var}'. Expected "
                    f"'{lake_abbreviation}_"
                    f"{surface_type}'."
                )

            lake = self.lake_lookup.get(
                lake_abv
            )

            if lake is None:

                raise ValueError(
                    f"Invalid lake abbreviation "
                    f"'{lake_abv}' in mask "
                    f"'{mask_var}'. Expected one "
                    f"of {list(self.lake_lookup.keys())}."
                )

            # ----------------------------------------------------------
            # Insert result into database
            # ----------------------------------------------------------

            self.db.add(
                cfs_run,
                forecast_year,
                forecast_month,
                lake,
                surface_type,
                config["output_name"],
                value.item(),
            )

    # ==================================================================
    # Processing: Mean
    # ==================================================================

    @staticmethod
    def process_mean(
        data,
        mask,
        area=None,
        num_days=None,
    ):
        """
        Calculate the mean value over the mask.

        Used for variables where the GRIB values should simply
        be averaged across the masked domain.

        Examples
        --------
        Surface pressure in Pa.
        2-m temperature in K.
        """

        masked_data = np.ma.masked_where(
            np.isnan(mask),
            data,
        )

        return np.mean(
            masked_data
        )

    # ==================================================================
    # Processing: Accumulated depth
    # ==================================================================

    @staticmethod
    def process_accumulated_depth(
        data,
        mask,
        area,
        num_days=None,
    ):
        """
        Calculate an area-weighted accumulated depth.

        Intended for variables such as runoff where the
        GRIB value is already an accumulated depth in kg/m2.

        For water:

            1 kg/m2 = 1 mm

        Therefore, no time conversion is required.
        """

        total = np.sum(
            data
            * mask
            * area
        )

        total_area = np.nansum(
            mask
            * area
        )

        if total_area == 0:
            raise ValueError(
                "Mask contains no valid area."
            )

        return (
            total
            / total_area
        )

    # ==================================================================
    # Processing: Rate to accumulation
    # ==================================================================

    @staticmethod
    def process_rate_to_accumulation(
        data,
        mask,
        area,
        num_days,
    ):
        """
        Convert a rate in kg/m2/s to a monthly
        accumulated depth in mm.

        This follows the same calculation currently
        used for precipitation in the original
        processing function.
        """

        seconds_in_day = (
            24
            * 60
            * 60
        )

        total = (
            np.sum(
                data
                * mask
                * area
            )
            * seconds_in_day
            * num_days
        )

        total_area = np.nansum(
            mask
            * area
        )

        if total_area == 0:
            raise ValueError(
                "Mask contains no valid area."
            )

        return (
            total
            / total_area
        )

    # ------------------------------------------------------------------

    @staticmethod
    def process_evaporation(
        data,
        mask,
        area,
        num_days,
    ):
        """
        Placeholder for evaporation processing.

        Evaporation requires latent heat flux and potentially
        temperature, so it is better handled as a specialized
        processor rather than treating it as a simple variable.
        """

        raise NotImplementedError(
            "Evaporation requires additional variables "
            "(e.g., latent heat flux and temperature). "
            "Implement this as a specialized processor."
        )

In [ ]:
variables = {
    "watr": {
        "output_name": "runoff",
        "long_name": "Runoff",
        "units": "kg/m2",
        "type_of_level": "surface",
        "level": None,
        "processor": "accumulated_depth",
    },

    "avg_sp": {
        "output_name": "surface_pressure",
        "long_name": "Surface Pressure",
        "units": "Pa",
        "type_of_level": "surface",
        "level": None,
        "processor": "mean",
    },

    "avg_2t": {
        "output_name": "air_temperature",
        "long_name": "2-m Air Temperature",
        "units": "K",
        "type_of_level": "heightAboveGround",
        "level": 2,
        "processor": "mean",
    },
}

In [ ]:
database = 'cfs_forecast_data.db'
table = 'cfs_forecast_data'

processor = CFSProcessing(database, table)

processor.process_files(
    download_dir="/path/to/cfs/files",
    mask_file="/path/to/mask.nc",
    mask_variables=[
        "sup_lake",
        "sup_land",
        "mih_lake",
        "mih_land",
        "eri_lake",
        "eri_land",
        "ont_lake",
        "ont_land",
    ],
    variables=variables,
)

In [ ]:
def process_files_new(self, download_dir, mask_file, mask_variables):
        """
        Process CFS GRIB files and insert lake-averaged variables into the database.

        This function loops through downloaded CFS GRIB files for a forecast run,
        extracts relevant variables, remaps them to the mask grid, calculates
        lake- or land-area weighted averages, and stores the results in the
        forecast database.

        Processed variables include:
            - precipitation from pgbf files
            - 2-meter air temperature from flxf files
            - evaporation from latent heat flux in flxf files

        Parameters
        ----------
        download_dir : str
            Directory containing downloaded CFS GRIB files.

        mask_file : str
            Path to the NetCDF mask file. The mask file must include:
                - latitude
                - longitude
                - lake/land mask variables listed in `mask_variables`

        mask_variables : list of str
            List of mask variable names to process.

            Expected format:
                "{lake_abbreviation}_{surface_type}"

            Examples:
                - "sup_lake"
                - "sup_land"
                - "mih_lake"
                - "eri_land"

            Valid lake abbreviations are:
                - "sup" -> "superior"
                - "mih" -> "michigan-huron"
                - "eri" -> "erie"
                - "ont" -> "ontario"

        Returns
        -------
        None
            Results are inserted directly into the database using `self.db.add()`.
        """

        # -------------------------
        # Validate inputs
        # -------------------------
        if not os.path.isdir(download_dir):
            raise ValueError("ERROR: The specified directory does not exist.")

        if not os.path.exists(mask_file):
            raise ValueError("ERROR: mask_file not found.")

        if not isinstance(mask_variables, list):
            raise ValueError("ERROR: mask_variables must be a list of strings.")

        # -------------------------
        # Load mask grid and areas
        # -------------------------
        # The mask grid defines the target latitude/longitude grid and lake/land masks.
        mask_ds = nc.Dataset(mask_file)

        mask_lat = mask_ds.variables["latitude"][:]
        mask_lon = mask_ds.variables["longitude"][:]

        # Calculate grid-cell area for area-weighted lake/land averages.
        area = calculate_grid_cell_areas(mask_lon, mask_lat)

        # Mapping from mask file lake abbreviations to full lake names.
        lake_lookup = {
            "eri": "erie",
            "ont": "ontario",
            "sup": "superior",
            "mih": "michigan-huron",
        }

        # -------------------------
        # Remove index files
        # -------------------------
        # cfgrib may create or use .idx files. Remove them so stale index files
        # do not interfere with reading newly downloaded GRIB files.
        for f in os.listdir(download_dir):
            if f.endswith(".idx"):
                os.remove(os.path.join(download_dir, f))

        # -------------------------
        # Process each GRIB file
        # -------------------------
        for filename in sorted(os.listdir(download_dir)):
            file = os.path.join(download_dir, filename)

            # File names are expected to contain the CFS run and forecast month.
            # Example structure depends on your downloaded CFS naming convention.
            parts = filename.split(".")

            cfs_run = parts[2]

            forecast_year = int(parts[3][:4])
            forecast_month = int(parts[3][4:6])

            # flxf files contain temperature and latent heat flux fields.
            if filename.startswith("flxf") and filename.endswith(".grib.grb2"):

                # -------------------------
                # Surface Pressure
                # -------------------------

                flx_surface = cfgrib.open_dataset(
                    file,
                    engine="cfgrib",
                    filter_by_keys={"typeOfLevel": "surface"},
                    decode_timedelta=False,
                )

                # Variable name differs between CFS data versions.
                pressure = flx_surface["avg_sp"]

                # Cut the pressure field to the mask extent.
                pressure_cut = pressure.sel(
                    latitude=slice(mask_lat.max(), mask_lat.min()),
                    longitude=slice(mask_lon.min(), mask_lon.max()),
                )

                # Remap pressure to the mask grid.
                pressure_remap = pressure_cut.interp(
                    latitude=mask_lat,
                    longitude=mask_lon,
                    method="linear",
                )

                for mask_var in mask_variables:
                    # Create a mask where valid mask cells are retained.
                    mask_ones = np.ma.masked_where(
                        np.isnan(mask_ds.variables[mask_var][:]),
                        np.ones_like(mask_ds.variables[mask_var][:]),
                    )

                    # Calculate mean pressure over the mask area.
                    pressure_avg = np.mean(pressure_remap * mask_ones)

                    lake_abv, surface_type = mask_var.split("_")
                    lake = lake_lookup.get(lake_abv)

                    if lake is None:
                        raise ValueError(
                            "ERROR: The mask variables need to begin with "
                            "'eri', 'ont', 'sup', or 'mih'. Check the mask file."
                        )

                    # Insert surface pressure into the database.
                    self.db.add(
                        cfs_run,
                        forecast_year,
                        forecast_month,
                        lake,
                        surface_type,
                        "surface_pressure",
                        pressure_avg.item(),
                    )

                # -------------------------
                # Runoff
                # -------------------------

                runoff = flx_surface["watr"]  # Runoff in kg/m^2

                # Cut runoff to the mask extent.
                runoff_cut = runoff.sel(
                    latitude=slice(mask_lat.max(), mask_lat.min()),
                    longitude=slice(mask_lon.min(), mask_lon.max()),
                )

                # Remap runoff to the mask grid.
                runoff_remap = runoff_cut.interp(
                    latitude=mask_lat,
                    longitude=mask_lon,
                    method="linear",
                )

                for mask_var in mask_variables:
                    mask = mask_ds.variables[mask_var][:]

                    # Calculate area-weighted monthly runoff avg.
                    total_runoff = np.sum(runoff_remap * area * mask)
                    runoff_mm = total_runoff / np.nansum(mask * area)

                    lake_abv, surface_type = mask_var.split("_")
                    lake = lake_lookup.get(lake_abv)

                    if lake is None:
                        raise ValueError(
                            "ERROR: The mask variables need to begin with "
                            "'eri', 'ont', 'sup', or 'mih'. Check the mask file."
                        )

                    # Insert runoff into the database.
                    self.db.add(
                        cfs_run,
                        forecast_year,
                        forecast_month,
                        lake,
                        surface_type,
                        "runoff",
                        runoff_mm.item(),
                    )

                    # -------------------------
                    # Snow Water Equivalent (SWE)
                    # -------------------------
                    swe = flx_surface["sdwe"] # SWE in kg/m^2

                    # Cut the SWE field to the mask extent.
                    swe_cut = swe.sel(
                        latitude=slice(mask_lat.max(), mask_lat.min()),
                        longitude=slice(mask_lon.min(), mask_lon.max()),
                    )

                    # Remap SWE to the mask grid.
                    swe_remap = swe_cut.interp(
                        latitude=mask_lat,
                        longitude=mask_lon,
                        method="linear",
                    )

                    for mask_var in mask_variables:
                        mask = mask_ds.variables[mask_var][:]

                        # Calculate area-weighted total SWE.
                        total_swe = np.sum(swe_remap * mask * area)
                        swe_mm = total_swe / np.nansum(mask * area)

                        lake_abv, surface_type = mask_var.split("_")
                        lake = lake_lookup.get(lake_abv)

                        if lake is None:
                            raise ValueError(
                                "ERROR: The mask variables need to begin with "
                                "'eri', 'ont', 'sup', or 'mih'. Check the mask file."
                            )

                        # Insert SWE into the database.
                        self.db.add(
                            cfs_run,
                            forecast_year,
                            forecast_month,
                            lake,
                            surface_type,
                            "snow_water_equivalent",
                            swe_mm.item(),
                        )

            # -------------------------
            # Skip files that do not match expected CFS GRIB patterns
            # -------------------------
            else:
                print(f"Skipping unrecognized file: {filename}")
                continue